In [1]:
import GA
import config
import qfm_lib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from math import comb

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, f1_score, balanced_accuracy_score,
    precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import mutual_info_classif
import shap
import dcor
import seaborn as sns
from scipy.spatial.distance import pdist, squareform

from itertools import combinations
from collections import Counter

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator

from qfm_lib import *

from sklearn.cluster import SpectralClustering

In [2]:
def load_dataset(dataset_dir):
    df = pd.read_csv(dataset_dir)
    df['Class'] = df['Class'].map({'NonToxic': 1,'Toxic': 0})
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

# feature reduction to match number of qubits in qpu
def reduce_features(X, y, n_features):
    if X.shape[1] <= n_features:
        return X
    else:
        mi = mutual_info_classif(X, y, n_neighbors=5, discrete_features=False, random_state=SEED)
    top_indices = np.argsort(mi)[-n_features:]
    return X[:, top_indices]


def data_preprocessing(X_train, X_test):
    scaler = StandardScaler() # Rescale the data using z-score: (x - μ (mean))/μ (standard deviation)
    X_tr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    return X_tr, Xte

def data_preprocessing_all(X):
    scaler = StandardScaler() # Rescale the data using z-score: (x - μ (mean))/μ (standard deviation)
    X_scaled = scaler.fit_transform(X)
    return X_scaled

In [3]:
def calculate_MI_matrix(X):
    num_features = X.shape[1]
    MI_b = np.zeros((num_features, num_features))
    for i in range(num_features):
        for j in range(i+1, num_features):
            MI_b[i, j] = mutual_info_regression(
                X[:, [i]],
                X[:, j],
                n_neighbors=5,
                random_state=SEED,
            )[0]
            MI_b[j, i] = MI_b[i, j]

    MI_norm = MI_b / (MI_b.max() + 1e-12)
    return MI_norm

def calculate_J_matrix(X):
    MI_matrix = calculate_MI_matrix(X)
    j = MI_matrix.copy()
    rho_thr = 0.0
    j[j < rho_thr] = 0.0
    np.fill_diagonal(j, 0.0)
    return j

In [17]:
def ds_curve(t, T):
    s = t / T
    D = -((np.pi**2) / (4 * T)) * np.sin(np.pi * s) * np.sin(
        np.pi * (np.sin((np.pi / 2) * s) ** 2)
    )
    return D


def s_curve(t_norm: float) -> float:
    return np.sin(0.5 * np.pi * np.sin(0.5 * np.pi * t_norm) ** 2) ** 2


def makeRt(h_x, J_dict, s, sum_hi2, sum_Jij2):
    sum_hi4 = 0
    sum_Jij4 = 0
    n = len(h_x)

    for hi in h_x:
        sum_hi4 += hi**4

    for (i, j), Jij in J_dict.items():
        sum_Jij4 += Jij**4

    sum_hi2Jii2 = 0
    for (i, j), Jij in J_dict.items():
        hi = h_x[i]
        hj = h_x[j]
        sum_hi2Jii2 += (hi**2 + hj**2) * (Jij**2)

    contrib_duplet = 2 * 6 * sum_hi2Jii2  # i<>j

    E = {(min(i, j), max(i, j)) for (i, j) in J_dict.keys()}

    def J(i, j):
        return J_dict[(i, j)] if (i, j) in J_dict else J_dict[(j, i)]

    sum_triplet = 0.0
    for i, j, k in combinations(range(n), 3):
        e1 = (min(i, j), max(i, j))
        e2 = (min(i, k), max(i, k))
        e3 = (min(j, k), max(j, k))

        if e1 in E and e2 in E and e3 in E:
            Jij = J(i, j)
            Jik = J(i, k)
            Jjk = J(j, k)
            sum_triplet += (
                Jij**2 * Jik**2 + Jij**2 * Jjk**2 + Jik**2 * Jjk**2
            )

    contrib_triplet = 6.0 * sum_triplet  # i<j<k

    Rt = ((1 - s) ** 2) * (sum_hi2 + 4 * sum_Jij2) + (s**2) * (
        sum_hi4 + 2 * sum_Jij4 + contrib_duplet + contrib_triplet
    )
    return Rt


def makeAlpha1(h_x, J_dict, s):
    sum_hi2 = 0
    sum_Jij2 = 0

    for hi in h_x:
        sum_hi2 += hi**2

    for (i, j), Jij in J_dict.items():
        sum_Jij2 += Jij**2

    alpha1 = -(1 / 4) * (sum_hi2 + 2 * sum_Jij2)
    Rt = makeRt(h_x, J_dict, s, sum_hi2, 2 * sum_Jij2)
    alpha1 = alpha1 / Rt
    return alpha1

In [20]:
def matrix_to_Jdict(J, tol=1e-12):
    """
    Convert symmetric J matrix to {(i, j): Jij} with i < j.
    """
    L = J.shape[0]
    J_dict = {}

    for i in range(L):
        for j in range(i + 1, L):
            if abs(J[i, j]) > tol:
                J_dict[(i, j)] = J[i, j]

    return J_dict

In [4]:
import quimb as qu
import quimb.tensor as qtn
from scipy.linalg import expm

In [125]:
def pauli_X():
    return np.array([[0, 1], [1, 0]])

def pauli_Y():
    return np.array([[0, -1j], [1j, 0]])

def pauli_Z():
    return np.array([[1, 0], [0, -1]])

def gate_Y(theta):
    Y = pauli_Y()
    return expm(1j * theta * Y)

def gate_YZ_ZY(theta):
    Y = pauli_Y()
    Z = pauli_Z()
    H = np.kron(Y, Z) + np.kron(Z, Y)
    return expm(1j * theta * H).reshape(2, 2, 2, 2)

def gen_cd_gates(x, J, t, T=0.005):
    L = len(x)

    # midpoint schedule
    t_mid = 0.25*t
    s = s_curve(t_mid / T)
    ds = ds_curve(t_mid, T)
    alpha1 = makeAlpha1(x, J, s)

    pref = -2.0 * ds * alpha1

    # --- initial |+> ---
    for i in range(L):
        yield ("H", i)

    # --- single-site CD ---
    for i, hi in enumerate(x):
        if hi != 0.0:
            yield (gate_Y(pref * hi), i)

    # --- two-body CD (i < j) ---
    for (i, j), Jij in J.items():
        if i < j and Jij != 0.0:
            yield (gate_YZ_ZY(pref * Jij), i, j)


def build_cd_circuit_mps(x, J, t, T=0.005, chi=32, cutoff=1e-10):

    L = len(x)

    circ = qtn.CircuitMPS.from_gates(
        gates=gen_cd_gates(x, J, t, T=T),
        N=L,
        max_bond=chi,
        cutoff=cutoff,
        progbar=True,
    )

    return circ, L

def measure_Z_all(circ, L):
    return np.array([circ.local_expectation(qu.pauli("Z"), (i,)) for i in range(L)])

def measure_ZZ_pairs(circ, L):
    vals = []
    for i in range(L):
        for j in range(i + 1, L):
            vals.append(circ.local_expectation(qu.pauli("Z") & qu.pauli("Z"), (i, j)))
    return np.array(vals)


In [ ]:
def compute_cd_features_dataset(X_c, J_dict, t, chi=None, T=1.0):
    n_samples = X_c.shape[0]
    L = X_c.shape[1]

    X_q = np.zeros((n_samples, L + L*(L-1)//2), dtype=np.float32)

    for s in range(n_samples):
        circ, _ = build_cd_circuit_mps(
            X_c[s],
            J_dict,
            t=t,
            T=T,
            chi=chi,
        )

        X_q[s] = measure_Z_all(circ, L)
        X_q[s + n_samples] = measure_ZZ_pairs(circ, L)

    return X_q

In [140]:
def build_quantum_features_all_tn(
    X,
    J_dict,
    tau,
    chi=16,
):
    """
    Tensor-network version of quantum feature generation.

    Runs MPS circuit per sample and measures:
        - <Z_i>
        - <Z_i Z_j>

    Returns:
        Xq_all : (N, n_obs)
        pairs_2q : list of (i, j)
    """

    N, n = X.shape

    # ----- prepare ZZ index list once -----
    pairs_2q = [(i, j) for i in range(n) for j in range(i + 1, n)]

    n_obs = n + len(pairs_2q)
    Xq_out = np.zeros((N, n_obs), dtype=np.float32)

    # ----- loop over samples (TN cannot batch) -----
    for s in range(N):

        x_vec = X[s]

        # --- build & run MPS CD circuit ---
        circ, L = build_cd_circuit_mps(
            x_vec,
            J_dict,
            t=tau,
            chi=chi,
        )

        # --- measure observables ---
        Z1 = measure_Z_all(circ, L)
        ZZ = measure_ZZ_pairs(circ, L)

        # --- concatenate into feature vector ---
        Xq_out[s] = np.real(np.concatenate([Z1, ZZ])).astype(np.float32)

    return Xq_out, pairs_2q

def compute_quantum_features(X, base_folder):
    X = data_preprocessing_all(X)
    n_b = X.shape[1]
    J = calculate_J_matrix(X)
    #J = calculate_J_matrix_dcor(X)
    #J = calculate_J_matrix_hsic(X)
    
    n = J.shape[0]
    J_dict = {}

    # only top part (i<j)
    for i in range(n):
        for j in range(i + 1, n):
            Jij = float(J[i, j])
            if Jij > 0.0:
                J_dict[(i, j)] = Jij

    print("Number of active 2-local interactions:", len(J_dict))
    tau = 0.005
    Xq_all_raw, pairs_2q = build_quantum_features_all_tn(
        X,
        J_dict,
        tau,
    )
        
    print('Quantum features successfully generated!')
    return Xq_all_raw, pairs_2q


In [129]:
path = "data.csv"
X_full, y_full = load_dataset(path)
if X_full is not None and y_full is not None:
    print("Dataset loaded successfully! with shape:", X_full.shape)

X_c = reduce_features(X_full, y_full, n_features=20)
# take just first 10 samples for testing
X_c = X_c[:10]
y = y[:10]

y = y_full
base_folder = (f"results_test")

Dataset loaded successfully! with shape: (171, 1203)


In [133]:
Xq_all, pairs_2q = compute_quantum_features(X_c, base_folder)

Number of active 2-local interactions: 109


max_bond=100, error~=0.603: 100%|##########| 149/149 [00:35<00:00,  4.21it/s]   


Quantum features successfully generated!


In [142]:
def init_metrics_dict():
    return {
        "AUC": [],
        "F1 Macro": [],
        "Precision Macro": [],
        "Recall Macro": [],
        "Accuracy": [],
    }

def update_metrics(metrics_dict, y_test, y_pred, y_proba):
    metrics_dict["AUC"].append(roc_auc_score(y_test, y_proba))
    metrics_dict["F1 Macro"].append(f1_score(y_test, y_pred, average="macro"))
    metrics_dict["Precision Macro"].append(precision_score(y_test, y_pred, average="macro", zero_division=0))
    metrics_dict["Recall Macro"].append(recall_score(y_test, y_pred, average="macro"))
    metrics_dict["Accuracy"].append(accuracy_score(y_test, y_pred))

def median_metrics(metrics_dict):
    return {k: np.median(v) for k, v in metrics_dict.items()}

m = 1
k_max = 2
q_ks_for_aug = [1, 2]  # which ks enter X_aug
tau = 0.005
l = 1  # order of approximation of CD terms
k_top = 50

def cross_validate_model(X_c, y, base_folder, n_splits=2, n_repeats=3):
    qc_rskf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=SEED)
    
    # Initialize metrics
    all_metrics = {}
    model_names = ["Classical_orig"] + [f"Q_{k}body" for k in range(1, k_max + 1)] + ["SHAP50"]
    COLS = ["AUC", "F1 Macro", "Precision Macro", "Recall Macro", "Accuracy"]
    for m in model_names:
        all_metrics[m] = {c: [] for c in COLS}
    
    shap_rankings = []

    # Classical features
    X_cl_all = data_preprocessing_all(X_c)
    n_samples, n_cl = X_cl_all.shape

    # Quantum features
    #Xq_all_raw, pairs_2q = load_quantum_features(base_folder, n_cl)
    Xq_all_raw, pairs_2q = compute_quantum_features(X_cl_all, base_folder)
    n_q_samples, n_q_feats = Xq_all_raw.shape

    n1 = X_cl_all.shape[1]
    n2 = n_q_feats - n1

    # Feature names
    cl_names = [f"cl_{j}" for j in range(n_cl)]
    q1_names = [f"q1_z_{i}" for i in range(n1)]
    q2_names = [f"q2_z_{i}_{j}" for (i, j) in pairs_2q] if n2 > 0 else []

    q_names_by_k_global = {1: q1_names}
    if k_max >= 2 and n2 > 0:
        q_names_by_k_global[2] = q2_names

    block_sizes = {1: n1}
    if k_max >= 2:
        block_sizes[2] = n2

    gamma_q = 1.0  # scaling factor for quantum features in X_aug

    # --------- SINGLE LOOP OVER FOLDS ---------
    for fold, (train_id, test_id) in enumerate(qc_rskf.split(X_cl_all, y)):
        print(f"\nFold {fold}")

        # Split classical
        X_train, X_test = data_preprocessing(X_cl_all[train_id], X_cl_all[test_id])
        y_train, y_test = y[train_id], y[test_id]

        # Split and scale quantum
        Xq_train_full = StandardScaler().fit_transform(Xq_all_raw[train_id])
        Xq_test_full = StandardScaler().fit_transform(Xq_all_raw[test_id])

        # Separate quantum blocks
        Xq_blocks_train = {}
        Xq_blocks_test = {}
        if k_max >= 1:
            Xq_blocks_train[1] = Xq_train_full[:, :n1]
            Xq_blocks_test[1] = Xq_test_full[:, :n1]
        if k_max >= 2 and n2 > 0:
            Xq_blocks_train[2] = Xq_train_full[:, n1 : n1 + n2]
            Xq_blocks_test[2] = Xq_test_full[:, n1 : n1 + n2]

        # ----------------- Classical Model -----------------
        clf_classical = GradientBoostingClassifier(n_estimators=1000, random_state=SEED)
        clf_classical.fit(X_train, y_train)
        y_pred = clf_classical.predict(X_test)
        y_proba = clf_classical.predict_proba(X_test)[:, 1]
        for metric, func in zip(COLS,
                                [roc_auc_score, lambda y, yp: f1_score(y, yp, average="macro"),
                                 lambda y, yp: precision_score(y, yp, average="macro", zero_division=0),
                                 lambda y, yp: recall_score(y, yp, average="macro"),
                                 accuracy_score]):
            all_metrics["Classical_orig"][metric].append(func(y_test, y_pred if metric!="AUC" else y_proba))

        # ----------------- Quantum Models -----------------
        for k in range(1, k_max + 1):
            if k not in Xq_blocks_train:
                continue
            Xqk_train, Xqk_test = Xq_blocks_train[k], Xq_blocks_test[k]
            clf_qk = GradientBoostingClassifier(n_estimators=1000, random_state=SEED)
            clf_qk.fit(Xqk_train, y_train)
            y_pred_q = clf_qk.predict(Xqk_test)
            y_proba_q = clf_qk.predict_proba(Xqk_test)[:, 1]
            for metric, func in zip(COLS,
                                    [roc_auc_score, lambda y, yp: f1_score(y, yp, average="macro"),
                                     lambda y, yp: precision_score(y, yp, average="macro", zero_division=0),
                                     lambda y, yp: recall_score(y, yp, average="macro"),
                                     accuracy_score]):
                all_metrics[f"Q_{k}body"][metric].append(func(y_test, y_pred_q if metric!="AUC" else y_proba_q))

        # ----------------- X_aug Model + SHAP -----------------
        q_ks_for_aug_eff = [k for k in q_ks_for_aug if k in Xq_blocks_train]
        Xq_train_aug = np.hstack([Xq_blocks_train[k] for k in q_ks_for_aug_eff])
        Xq_test_aug = np.hstack([Xq_blocks_test[k] for k in q_ks_for_aug_eff])

        X_aug_train = np.hstack([X_train, gamma_q * Xq_train_aug]).astype(np.float32)
        X_aug_test = np.hstack([X_test, gamma_q * Xq_test_aug]).astype(np.float32)
        all_q_names_for_aug = [name for k in q_ks_for_aug_eff for name in q_names_by_k_global[k]]
        feat_names_aug = cl_names + all_q_names_for_aug

        model_shap = GradientBoostingClassifier(n_estimators=1000, random_state=SEED)
        model_shap.fit(X_aug_train, y_train)
        explainer = shap.TreeExplainer(model_shap)
        shap_values = explainer.shap_values(X_aug_train)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
        
        # ----------------- SHAP Sorting and Saving -----------------
        # Split SHAP: classical features
        offset = 0
        shap_cl = mean_abs_shap[offset : offset + n_cl]
        offset += n_cl

        # SHAP of quantum features by k
        shap_by_k = {}
        for k in q_ks_for_aug_eff:
            size_k = block_sizes[k]
            shap_by_k[k] = mean_abs_shap[offset : offset + size_k]
            offset += size_k

        # reorder quantum features by SHAP importance
        Xq_blocks_train_sorted = {}
        Xq_blocks_test_sorted = {}
        q_names_sorted_by_k = {}

        for k in q_ks_for_aug_eff:
            shap_k = shap_by_k[k]
            names_k = q_names_by_k_global[k]
            idx_sort = np.argsort(shap_k)[::-1]

            Xq_blocks_train_sorted[k] = Xq_blocks_train[k][:, idx_sort]
            Xq_blocks_test_sorted[k] = Xq_blocks_test[k][:, idx_sort]
            q_names_sorted_by_k[k] = [names_k[i] for i in idx_sort]

        Xq_train_sorted = np.hstack([Xq_blocks_train_sorted[k] for k in q_ks_for_aug_eff])
        Xq_test_sorted = np.hstack([Xq_blocks_test_sorted[k] for k in q_ks_for_aug_eff])
        q_names_sorted_all = [name for k in q_ks_for_aug_eff for name in q_names_sorted_by_k[k]]

        # Save per-fold quantum features CSV
        df_q_train = pd.DataFrame(Xq_train_sorted, columns=q_names_sorted_all)
        df_q_train["y"] = y_train
        df_q_train["split"] = fold
        df_q_train["set"] = "train"
        df_q_train.to_csv(f"{base_folder}/fold{fold}_train.csv", index=False)

        df_q_test = pd.DataFrame(Xq_test_sorted, columns=q_names_sorted_all)
        df_q_test["y"] = y_test
        df_q_test["split"] = fold
        df_q_test["set"] = "test"
        df_q_test.to_csv(f"{base_folder}/fold{fold}_test.csv", index=False)

        # Save SHAP rankings per fold
        shap_df_fold = pd.DataFrame({
            "split": fold,
            "feature": feat_names_aug,
            "mean_abs_shap": mean_abs_shap
        }).sort_values("mean_abs_shap", ascending=False)
        shap_rankings.append(shap_df_fold)
        
        #shap_rankings.append(pd.DataFrame({"split": fold, "feature": feat_names_aug, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False))

        # SHAP50 top features
        idx_top = np.argsort(mean_abs_shap)[-k_top:]
        X_train_50, X_test_50 = X_aug_train[:, idx_top], X_aug_test[:, idx_top]
        clf_shap50 = GradientBoostingClassifier(n_estimators=1000, random_state=SEED)
        clf_shap50.fit(X_train_50, y_train)
        y_pred_50, y_proba_50 = clf_shap50.predict(X_test_50), clf_shap50.predict_proba(X_test_50)[:, 1]
        for metric, func in zip(COLS,
                                [roc_auc_score, lambda y, yp: f1_score(y, yp, average="macro"),
                                 lambda y, yp: precision_score(y, yp, average="macro", zero_division=0),
                                 lambda y, yp: recall_score(y, yp, average="macro"),
                                 accuracy_score]):
            all_metrics["SHAP50"][metric].append(func(y_test, y_pred_50 if metric!="AUC" else y_proba_50))

    # ----------------- Convert to DataFrame -----------------
    df_all_list = []
    for m in model_names:
        df = pd.DataFrame(all_metrics[m])
        df["model_type"] = m
        df_all_list.append(df)
    df_all = pd.concat(df_all_list, ignore_index=True)
    shap_rankings_all = pd.concat(shap_rankings, ignore_index=True)
    shap_rankings_all.to_csv(
        f"{base_folder}/shap_rankings_k{k_max}_l{l}_T_{tau}_m_{m}_l_{l}_top{k_top}_rescaled_{n_cl}.csv",
        index=False,
    )

    df_all.to_csv(f"{base_folder}/cd_terms_qc_vs_classical_DIGITAL_k{k_max}_l{l}_T_{tau}_m_{m}_l_{l}_top{k_top}_rescaled_{n_cl}.csv", index=False)
    print(f"Saved in {base_folder}/cd_terms_qc_vs_classical_DIGITAL_k{k_max}_l{l}_T_{tau}_m_{m}_l_{l}_top{k_top}_rescaled_{n_cl}.csv")

    # ----------------- PRINT METRICS CONSISTENT WITH df_all -----------------
    metrics_order = ["F1 Macro", "Precision Macro", "Recall Macro", "AUC", "Accuracy"]
    model_order = ["Classical_orig"] + [f"Q_{k}body" for k in range(1, k_max + 1)] + ["SHAP50"]

    agg = df_all.groupby("model_type")[metrics_order].agg(["mean", "std"])

    print("\n=============MEAN================")
    for m in model_order:
        print(f"\n=== {m} ===")
        for metric in metrics_order:
            mean_val = agg.loc[m, (metric, "mean")]
            std_val = agg.loc[m, (metric, "std")]
            print(f"{metric}: {mean_val:.4f} ± {std_val:.4f}")

    print("\n=============MEDIAN================")
    agg_median = df_all.groupby("model_type")[metrics_order].median()
    for m in model_order:
        print(f"\n=== {m} ===")
        for metric in metrics_order:
            median_val = agg_median.loc[m, metric]
            # use same std as before for consistency
            std_val = agg.loc[m, (metric, "std")]
            print(f"{metric}: {median_val:.4f} ± {std_val:.4f}")

    return df_all, shap_rankings_all

In [143]:
df_all, shap_rankings_all = cross_validate_model(X_c, y, base_folder, n_splits=2, n_repeats=3)

Number of active 2-local interactions: 109


max_bond=16, error~=0.955: 100%|##########| 149/149 [00:00<00:00, 200.62it/s]  


Quantum features successfully generated!


ValueError: Found input variables with inconsistent numbers of samples: [10, 171]